# Hands-On: Multi-Agent Travel Planner

In this exercise you will build a multi-agent system using LangGraph. The system acts as a travel agency with:

- An **orchestrator** agent that talks to the user and delegates to specialists
- A **plane agent** that searches for flights using a tool
- An **accommodations agent** that searches for hotels using a tool
- A **summary agent** that produces the final travel plan

You will implement:
1. The LangGraph state
2. The agent nodes (functions that call the LLM)
3. The routing function (decides which agent to call next)
4. The full graph with nodes, edges, and conditional edges

The system prompts and tools are provided. Your job is to wire everything together.

## Step 0: Setup
Run the cell below to import all dependencies and initialise the LLM client. Nothing to change here.

In [1]:
import langgraph
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image, display

from config import Config
from pydantic import BaseModel, Field
from typing import Annotated
from langgraph.graph.message import add_messages
from prompts import BOSS_AGENT_PROMPT, PLANE_AGENT_PROMPT, ACCOMMODATIONS_AGENT_PROMPT, ITINERARY_AGENT_PROMPT
import gradio as gr
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool

config = Config()
llm = config.get_llm()

/Users/rodrigo/Desktop/work_projects/AI_training/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Define the State
The state is the shared data structure that all nodes in the graph can read and write to. It flows through the entire graph.

For this system, the state needs at minimum:
- `messages`: the conversation history (use `add_messages` reducer so new messages are appended, not replaced)

You can add extra fields like `origin` and `destination` if you want to track them explicitly.

**TODO:** Define a Pydantic `State` class with the fields described above.

Hint: use `Annotated[list, add_messages]` for the messages field.

In [ ]:
# TODO: Define the State class and its attributes
class State(BaseModel):
    pass

## Step 2: Tools (Provided)

The tools below simulate external APIs for searching flights and hotels. They are already implemented for you using the `@tool` decorator from LangChain.

Notice how each tool has:
- A clear function name
- A docstring (this is what the LLM reads to decide when to use the tool)
- Type-annotated parameters
- A return value

Run the cell below. Nothing to change here.

In [3]:
@tool
def search_flights(origin: str, destination: str, date: str) -> list[dict]:
    """Search for available flights between two cities on a given date."""
    return [
        {"airline": "Air France", "departure": "08:30", "arrival": "14:45", "stops": 1, "price": 420},
        {"airline": "Iberia", "departure": "11:00", "arrival": "16:30", "stops": 0, "price": 530},
        {"airline": "Lufthansa", "departure": "06:15", "arrival": "13:00", "stops": 1, "price": 380},
    ]

@tool
def search_hotels(destination: str, checkin: str, checkout: str) -> list[dict]:
    """Search for hotels in a destination."""
    return [
        {"name": "Hotel Central", "location": "City center", "nightly_rate": 95, "rating": 4.2},
        {"name": "Grand Palace", "location": "Old town", "nightly_rate": 150, "rating": 4.7},
        {"name": "Budget Inn", "location": "Near station", "nightly_rate": 55, "rating": 3.8},
    ]

## Step 3: Define Agent Nodes

Each agent in the graph is a Python function (a "node") that:
1. Takes the current `State` as input
2. Prepends the agent's system prompt to the message history
3. Calls the LLM
4. Returns a dict with the new messages to add to state

You need to implement 4 agent nodes:

- `orchestrator_node` — Uses `BOSS_AGENT_PROMPT`. Calls `llm.invoke()` directly (no tools).
- `plane_agent_node` — Uses `PLANE_AGENT_PROMPT`. Must bind the `search_flights` tool to the LLM using `llm.bind_tools([search_flights])`.
- `accommodations_agent_node` — Uses `ACCOMMODATIONS_AGENT_PROMPT`. Must bind the `search_hotels` tool using `llm.bind_tools([search_hotels])`.
- `summary_agent_node` — Uses `ITINERARY_AGENT_PROMPT`. Calls `llm.invoke()` directly (no tools).

**TODO:** Implement the 4 node functions.

Each function should:
```python
def some_node(state: State) -> dict:
    messages = [{"role": "system", "content": SOME_PROMPT}] + state.messages
    response = llm.invoke(messages)  # or llm_with_tools.invoke(messages)
    return {"messages": [response]}
```

In [ ]:
# TODO: Implement the orchestrator node
def orchestrator_node(state: State) -> dict:
    pass


# TODO: Implement the plane agent node (bind search_flights tool)
def plane_agent_node(state: State) -> dict:
    pass


# TODO: Implement the accommodations agent node (bind search_hotels tool)
def accommodations_agent_node(state: State) -> dict:
    pass


# TODO: Implement the summary agent node
def summary_agent_node(state: State) -> dict:
    pass

## Step 4: Implement the Routing Function

The orchestrator agent decides which specialist to delegate to by including a keyword in its response (e.g. `HANDOFF:PLANE`). Your routing function must inspect the last message and return the name of the next node.

The routing function receives the current `State` and must return a **string** — the name of the next node to execute:

| Keyword in message | Return value |
|---|---|
| `HANDOFF:PLANE` | `"plane_agent"` |
| `HANDOFF:ACCOMMODATIONS` | `"accommodations_agent"` |
| `HANDOFF:SUMMARY` | `"summary_agent"` |
| None of the above | `END` (the conversation pauses, waiting for user input) |

**TODO:** Implement the `route_from_orchestrator` function.

Hint: get the last message content with `state.messages[-1]` and check if it contains the keywords.

In [ ]:
#TODO: Implement the route_from_orchestrator function to determine the next node based on the state
def route_from_orchestrator(state: State) -> str:

    #TODO: get last message from state

    #TODO: get the content of the last message

    #TODO: make conditional statements with keywords found in the message content, and return the name of the corresponding node to route to

    #TODO: return END to stay in the conversation with the user
    pass

## Step 5: Build the Graph

Now wire everything together using LangGraph's `StateGraph`. You need to:

1. **Create the graph** with your `State` class
2. **Add nodes** — one for each agent + one `ToolNode` per tool:
   - `"orchestrator"` → `orchestrator_node`
   - `"plane_agent"` → `plane_agent_node`
   - `"plane_search_tool"` → `ToolNode(tools=[search_flights])`
   - `"accommodations_agent"` → `accommodations_agent_node`
   - `"accommodations_search_tool"` → `ToolNode(tools=[search_hotels])`
   - `"summary_agent"` → `summary_agent_node`
3. **Add edges**:
   - `START` → `"orchestrator"` (entry point)
   - `"summary_agent"` → `END`
   - `"plane_search_tool"` → `"plane_agent"` (tool result goes back to agent)
   - `"accommodations_search_tool"` → `"accommodations_agent"`
4. **Add conditional edges**:
   - From `"orchestrator"` → use your `route_from_orchestrator` function
   - From `"plane_agent"` → use `tools_condition` to decide: call tool or return to orchestrator
   - From `"accommodations_agent"` → use `tools_condition` to decide: call tool or return to orchestrator
5. **Compile** with `MemorySaver()` as the checkpointer

**TODO:** Build and compile the graph.

Hint for conditional edges with `tools_condition`:
```python
graph.add_conditional_edges("plane_agent", tools_condition, {"tools": "plane_search_tool", END: "orchestrator"})
```
This means: if the LLM response contains tool calls, go to the tool node. Otherwise, go back to orchestrator.

In [ ]:
# TODO: Build the graph
memory = MemorySaver()
graph = StateGraph(State)

# Add nodes


# Add edges


# Add conditional edges


# Compile
app = graph.compile(checkpointer=memory)

## Step 6: Visualise the Graph

If your graph is built correctly, the cell below will render it. This is a good checkpoint to verify your structure before running the chat.

The graph should show:
- `orchestrator` at the top with conditional edges to the 3 specialists and to END
- Each tool-using agent (`plane_agent`, `accommodations_agent`) should loop through its respective tool node
- `summary_agent` should go directly to END

In [ ]:
display(Image(app.get_graph().draw_mermaid_png()))

## Step 7: Chat Interface

The cells below set up a Gradio chat interface and the streaming function. Nothing to change here — just run them once your graph compiles.

Try a conversation like:
1. "I want to travel from Madrid to Paris"
2. Answer the orchestrator's questions about dates, budget, interests
3. Confirm the trip details
4. Watch the agents collaborate

In [ ]:
thread_config = {"configurable": {"thread_id": "1"}}


def chat_stream(user_input: str, history) -> str:
    for event in app.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        config=thread_config,
        stream_mode="updates",
    ):
        # event is a dict like {"orchestrator": {"messages": [...]}}
        for node_name, updates in event.items():
            print(f"--- {node_name} ---")
            for msg in updates.get("messages", []):
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    print(f"  Tool calls: {msg.tool_calls}")
                elif hasattr(msg, "content"):
                    print(f"  {msg.content[:200]}")
    # Return all assistant messages the user should see (skip tool/internal messages)
    state = app.get_state(thread_config)
    all_msgs = state.values["messages"]
    # Collect new assistant messages produced in this turn
    # The user's message is already tracked by Gradio, so only return assistant content
    new_responses = []
    for m in reversed(all_msgs):
        if hasattr(m, "role") and m.role == "user":
            break
        if hasattr(m, "type") and m.type == "human":
            break
        if hasattr(m, "content") and m.content and not (hasattr(m, "tool_calls") and m.tool_calls):
            role = getattr(m, "role", getattr(m, "type", ""))
            if role in ("assistant", "ai"):
                new_responses.append(m.content)
    new_responses.reverse()
    return "\n\n".join(new_responses) if new_responses else all_msgs[-1].content

In [ ]:
gr.ChatInterface(chat_stream).launch()